# IMF World Economic Outlook 9.0.0: Professional Economic Analytics Report

        This notebook documents the data engineering and exploratory analysis behind the Streamlit dashboard. The IMF WEO file is a wide annual time-series dataset: each row is a country/group plus an economic indicator, and each annual column from 1980 to 2031 contains observations or projections.

        The workflow below is designed for reproducibility:

        1. Dataset loading
        2. Schema inspection
        3. Cleaning and quality diagnostics
        4. Long-format transformation using `pandas.melt()`
        5. Economic EDA
        6. Required visualizations
        7. Automated insight generation
        8. Economic findings summary

## 1. Dataset Loading

        The notebook imports the reusable dashboard preprocessing layer instead of creating a separate cleaning path. This keeps the Streamlit app and analyst report aligned.

In [ ]:
from pathlib import Path
        import sys

        import matplotlib.pyplot as plt
        import pandas as pd
        import seaborn as sns

        PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
        sys.path.append(str(PROJECT_DIR))

        from filters import load_data, profile_dataset, detect_year_columns
        from charts import set_chart_theme, ranking_table, anomaly_detector, choose_scatter_pairs

        set_chart_theme()
        bundle = load_data(str(PROJECT_DIR / "data"))
        raw = bundle.raw
        wide = bundle.wide
        long_df = bundle.long
        profile = profile_dataset(bundle)

        profile

## 2. Schema Inspection

        The WEO schema contains rich metadata columns and annual time-series columns. Annual columns are detected automatically by checking numeric column names in the expected year range.

In [ ]:
print(f"Raw shape: {raw.shape}")
        print(f"Metadata columns: {len(bundle.metadata_columns)}")
        print(f"Year columns: {len(bundle.year_columns)}")
        print(f"Year coverage: {min(bundle.year_columns, key=int)} to {max(bundle.year_columns, key=int)}")

        schema_summary = pd.DataFrame({
            "column": raw.columns,
            "non_null": raw.notna().sum().values,
            "missing_pct": (raw.isna().mean().values * 100).round(2),
            "sample_value": [raw[col].dropna().iloc[0] if raw[col].notna().any() else None for col in raw.columns],
        })
        schema_summary.head(15)

## 3. Data Quality Diagnostics

        The diagnostics focus on missingness, duplicate rows, metadata completeness, year-column detection, and numeric conversion reliability.

In [ ]:
important_metadata = [
            "DATASET", "SERIES_CODE", "OBS_MEASURE", "COUNTRY", "INDICATOR", "FREQUENCY",
            "SCALE", "UNIT", "TOPIC", "KEY_INDICATOR", "SERIES_NAME", "FULL_DESCRIPTION",
            "PRIMARY_DOMESTIC_CURRENCY", "LATEST_ACTUAL_ANNUAL_DATA"
        ]
        metadata_quality = (
            wide[[col for col in important_metadata if col in wide.columns]]
            .isna()
            .mean()
            .mul(100)
            .round(2)
            .rename("missing_pct")
            .reset_index()
            .rename(columns={"index": "metadata_field"})
        )
        metadata_quality

In [ ]:
quality_checks = pd.DataFrame([
            {"check": "Full duplicate rows", "value": wide.duplicated().sum()},
            {"check": "Unique countries/groups", "value": wide["COUNTRY"].nunique()},
            {"check": "Unique indicators", "value": wide["INDICATOR"].nunique()},
            {"check": "Unique topics", "value": wide["TOPIC"].nunique()},
            {"check": "Valid long-format observations", "value": long_df["Value"].notna().sum()},
            {"check": "Numeric parse failures", "value": profile["numeric_parse_failures"]},
            {"check": "Missing annual values (%)", "value": round(profile["missing_year_value_pct"], 2)},
        ])
        quality_checks

## 4. Long Format Transformation

        The core transformation uses `pandas.melt()` to convert annual columns into `Year` and `Value`, while retaining IMF metadata for filtering, labeling, and economic context.

In [ ]:
long_preview = long_df[[
            "Country", "Indicator", "Year", "Value", "Topic", "Scale", "Unit", "Year Type", "Series Code"
        ]].dropna(subset=["Value"]).head(12)
        long_preview

In [ ]:
print(f"Long shape including missing values: {long_df.shape}")
        print(f"Valid numeric observations: {long_df['Value'].notna().sum():,}")
        print(f"Country-indicator-year duplicates: {long_df.duplicated(subset=['Country', 'Indicator', 'Year']).sum():,}")

## 5. Exploratory Economic Analysis

        The IMF WEO dataset is dominated by GDP, fiscal sector, consumer prices, current account, trade, population, unemployment, exchange rates, and debt indicators.

In [ ]:
topic_counts = (
            wide.groupby("TOPIC")["SERIES_CODE"]
            .count()
            .sort_values(ascending=False)
            .head(15)
        )
        topic_counts

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
        sns.barplot(x=topic_counts.values, y=topic_counts.index, ax=ax, color="#0E7490")
        ax.set_title("Top IMF WEO Topics by Series Count", loc="left", fontweight="bold")
        ax.set_xlabel("Series count")
        ax.set_ylabel("Topic")
        plt.tight_layout()
        plt.show()

## 6. Required Visualizations

        The dashboard implements the course-required chart types in an economic context: category distribution, histogram, line trend, ranking bar, scatter relationship, box plot, heatmap, area chart, count plot, and violin distribution.

In [ ]:
gdp_indicator = "Gross domestic product (GDP), Current prices, US dollar"
        inflation_indicator = "All Items, Consumer price index (CPI), Period average, percent change"
        growth_indicator = "Gross domestic product (GDP), Constant prices, Percent change"
        selected_countries = ["United States", "China, People's Republic of", "India", "Germany", "Japan"]
        analysis_df = long_df[
            long_df["Country"].isin(selected_countries)
            & long_df["Year"].between(2000, 2031)
            & long_df["Indicator"].isin([gdp_indicator, inflation_indicator, growth_indicator])
        ].dropna(subset=["Value"])
        analysis_df.head()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
        trend = analysis_df[analysis_df["Indicator"].eq(gdp_indicator)]
        sns.lineplot(data=trend, x="Year", y="Value", hue="Country", linewidth=2.2, ax=ax)
        ax.axvspan(2026, 2031, color="#E2E8F0", alpha=0.6, label="Forecast window")
        ax.set_title("GDP Trend Comparison, Current US Dollars", loc="left", fontweight="bold")
        ax.set_xlabel("Year")
        ax.set_ylabel("US dollars, billions")
        plt.tight_layout()
        plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        sns.histplot(data=analysis_df[analysis_df["Indicator"].eq(inflation_indicator)], x="Value", hue="Country", bins=25, ax=axes[0])
        axes[0].set_title("Inflation Distribution", loc="left", fontweight="bold")
        axes[0].set_xlabel("Percent change")

        gdp_2025 = ranking_table(long_df, gdp_indicator, 2025, 10, include_aggregates=False)
        sns.barplot(data=gdp_2025.sort_values("Value"), x="Value", y="Country", ax=axes[1], color="#2563EB")
        axes[1].set_title("Top GDP Economies, 2025", loc="left", fontweight="bold")
        axes[1].set_xlabel("US dollars, billions")
        axes[1].set_ylabel("")
        plt.tight_layout()
        plt.show()

In [ ]:
pairs = choose_scatter_pairs(long_df["Indicator"].dropna().unique())
        x_indicator, y_indicator = pairs["GDP vs Inflation"]
        scatter_source = long_df[
            long_df["Year"].eq(2025)
            & long_df["Indicator"].isin([x_indicator, y_indicator])
            & (~long_df["Is Aggregate"])
        ].dropna(subset=["Value"])
        scatter_pivot = scatter_source.pivot_table(index="Country", columns="Indicator", values="Value", aggfunc="mean").dropna().reset_index()

        fig, ax = plt.subplots(figsize=(8, 5))
        sns.scatterplot(data=scatter_pivot, x=x_indicator, y=y_indicator, ax=ax, color="#0F766E", s=65)
        sns.regplot(data=scatter_pivot, x=x_indicator, y=y_indicator, ax=ax, scatter=False, color="#334155")
        ax.set_title("GDP vs Inflation, 2025", loc="left", fontweight="bold")
        plt.tight_layout()
        plt.show()

In [ ]:
correlation_source = long_df[
            long_df["Year"].between(2020, 2025)
            & long_df["Indicator"].isin([gdp_indicator, inflation_indicator, growth_indicator])
        ].dropna(subset=["Value"])
        corr_pivot = correlation_source.pivot_table(index=["Country", "Year"], columns="Indicator", values="Value", aggfunc="mean")

        fig, ax = plt.subplots(figsize=(7, 5))
        sns.heatmap(corr_pivot.corr(), annot=True, fmt=".2f", cmap="vlag", center=0, ax=ax)
        ax.set_title("Economic Indicator Correlation Matrix", loc="left", fontweight="bold")
        plt.tight_layout()
        plt.show()

## 7. Automated Insight Generation

        The dashboard includes a robust anomaly detector that compares each country to the cross-sectional median for the same year using a median absolute deviation style score.

In [ ]:
anomalies = anomaly_detector(long_df[long_df["Year"].between(2000, 2031)], inflation_indicator, limit=10)
        anomalies

In [ ]:
latest_gdp = ranking_table(long_df, gdp_indicator, 2025, 10, include_aggregates=False)
        leader = latest_gdp.iloc[0]
        print(f"GDP leader in 2025: {leader['Country']} with {leader['Value']:,.2f} {leader['Scale']} {leader['Unit']}.")

        inflation_volatility = (
            long_df[long_df["Indicator"].eq(inflation_indicator) & long_df["Year"].between(2000, 2025)]
            .dropna(subset=["Value"])
            .groupby("Country")["Value"]
            .std()
            .sort_values(ascending=False)
            .head(10)
        )
        inflation_volatility

## 8. Economic Findings Summary

        Key findings from this dataset profile:

        - The IMF WEO file is already well-structured for macroeconomic analysis once the annual columns are melted into long format.
        - There are no full duplicate rows and no numeric parsing failures in the annual columns.
        - Missingness is concentrated in specialized metadata fields and early historical years, which is expected for international macroeconomic coverage.
        - Forecast years are available through 2031 and should be visually separated from historical observations.
        - GDP, inflation, growth, population, debt, current account, and fiscal sector indicators provide enough breadth for ranking, trend, anomaly, and correlation analysis.

        The Streamlit dashboard operationalizes these findings into an interactive BI-style product for economists and analysts.